# Experiment 4: Model Size Generalization

Does SoftGear's benefit generalize across model sizes?

**Prerequisites**: Runtime > Change runtime type > **GPU (T4)**

| Size | hidden_dim | ffn_dim | ~Parameters (7 gears) |
|------|-----------|---------|----------------------|
| small | 64 | 256 | ~1/4x |
| medium | 128 | 512 | 1x (= Exp1 baseline) |

We run gradual / none / from_scratch at "small" size and compare
with Exp1 results (medium) to check whether the relative ranking
of strategies is preserved.

## Setup

In [ ]:
!pip install -q git+https://github.com/byExist/softgear.git

In [ ]:
!softgear download sudoku9

In [ ]:
import torch
assert torch.cuda.is_available(), "GPU not available! Change runtime type."
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# type: ignore
from google.colab import drive

drive.mount("/content/drive")
CKPT_ROOT = "/content/drive/MyDrive/softgear/checkpoints"

In [ ]:
COMMON = "--task sudoku9 --num-gears 7 --patience 5 --max-total-steps 40000 --max-samples 50000 --seed 42"

## Training — Small (hidden_dim=64, ffn_dim=256)

In [ ]:
SMALL = "--hidden-dim 64 --ffn-dim 256 --num-heads 4"

In [ ]:
# gradual (SoftGear)
!softgear train --hardening gradual --checkpoint-dir {CKPT_ROOT}/exp4-small-gradual $SMALL $COMMON

In [ ]:
# none (no protection)
!softgear train --hardening none --checkpoint-dir {CKPT_ROOT}/exp4-small-none $SMALL $COMMON

In [ ]:
# from_scratch (all gears at once)
!softgear train --hardening from_scratch --checkpoint-dir {CKPT_ROOT}/exp4-small-scratch $SMALL $COMMON

## Results

Evaluate small models and compare with Exp1 (medium) checkpoints.

In [ ]:
import torch
import pandas as pd
from typing import Any
from softgear.tasks.registry import get_task
from softgear.config import DataConfig, ModelConfig

task = get_task("sudoku9")
data_cfg = DataConfig(path="data/sudoku-extreme", batch_size=64, num_workers=2, max_samples=None, curriculum=False)
_, val_loader = task.build_loaders(data_cfg)
device = torch.device("cuda")

experiments = {
    # Small (hidden_dim=64, ffn_dim=256)
    "small gradual": f"{CKPT_ROOT}/exp4-small-gradual",
    "small none": f"{CKPT_ROOT}/exp4-small-none",
    "small scratch": f"{CKPT_ROOT}/exp4-small-scratch",
    # Medium (hidden_dim=128, ffn_dim=512) — from Exp1
    "medium gradual": f"{CKPT_ROOT}/exp1-gradual",
    "medium none": f"{CKPT_ROOT}/exp1-none",
    "medium scratch": f"{CKPT_ROOT}/exp1-scratch",
}

results: dict[str, Any] = {}
for name, ckpt_dir in experiments.items():
    ckpt = torch.load(f"{ckpt_dir}/best.pt", map_location=device, weights_only=False)
    cfg = ckpt["config"]

    model_cfg = ModelConfig(**cfg["model"])
    model = task.build_model(model_cfg)
    task.mount_all_gears(model, model_cfg)
    model.load_state_dict(ckpt["model_state_dict"])
    model.to(device)
    model.eval()

    all_preds: list[torch.Tensor] = []
    all_targets: list[torch.Tensor] = []
    all_inputs: list[torch.Tensor] = []
    with torch.no_grad():
        for batch in val_loader:
            x, y = batch[0].to(device), batch[1].to(device)
            preds = task.predict_fn(model(x).logits)
            all_preds.append(preds)
            all_targets.append(y)
            all_inputs.append(x)

    metrics = task.metrics_fn(torch.cat(all_preds), torch.cat(all_targets), torch.cat(all_inputs))
    results[name] = {
        "hidden_dim": cfg["model"]["hidden_dim"],
        "ffn_dim": cfg["model"]["ffn_dim"],
        "phase": ckpt["phase"],
        "global_step": ckpt.get("global_step", "?"),
        "val_loss": ckpt["best_val_loss"],
        **metrics,
    }

df = pd.DataFrame(results).T
df.index.name = "condition"
df